## Image preparation

The below code can be used to transform the images in an input directory (Input_Dir) 
to the right size (e.g. 32x32 pixels) into an output directory (Output_Dir).

Note: Duplicates will be removed by evaluating the file hash

### Basic Parameter

Note: Do not rename any of the variables in this section, because the they are externally used in github action `Train Model`

In [ ]:
# Parameters
Input_Dir = 'data_raw_all'
Output_Dir = 'data_resize_all'

# Target image size [x, y, channels]
Target_Shape = (32, 32, 3)


### Load libraries and defaults

In [ ]:
import glob
import os

from pathlib import Path
import hashlib
import tensorflow as tf


### Delete output directory

In [ ]:
files = glob.glob(Output_Dir + '/*.jpg')
for f in files:
    os.remove(f)
print(str(len(files)) + " files have been deleted.")


### Load files and resize

In [ ]:
files = glob.glob(Input_Dir + '/*.jpg')
hashes = {}

for i, aktfile in enumerate(files):
    if i % 500 == 0:
        print(i, aktfile)

    # Read
    image_bytes = tf.io.read_file(aktfile)
    image = tf.image.decode_image(image_bytes, channels=Target_Shape[2], expand_animations=False)

    # Compute SHA256 hash of image content only, not of file itself (exclude metadata, etc)
    hash_val = hashlib.sha256(tf.io.encode_jpeg(image).numpy()).hexdigest()
    if hash_val in hashes:
        hashes[hash_val].append(aktfile)
        continue
    else:
        hashes[hash_val] = [aktfile]
    
    # Resize
    image = tf.image.resize(image, [Target_Shape[0], Target_Shape[1]], method=tf.image.ResizeMethod.MITCHELLCUBIC)
    image = tf.cast(image, tf.uint8)
    image = tf.io.encode_jpeg(image, quality=100)

    # Save
    base = os.path.basename(aktfile)
    save_path = os.path.join(Output_Dir, base)
    tf.io.write_file(save_path, image)



### Remove duplicate files

In [ ]:
# duplicate files are a risk to the metrics, they pollute the validation dataset
for hash in hashes:
    if len(hashes[hash]) > 1:
        print(f"Duplicates: {hashes[hash]}")    
        for duplicate in hashes[hash][1:]:
            # remove all except the first
            print(f"Removed: {duplicate}")
            os.remove(duplicate)    
